In [6]:
!pip install hcipy
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.8/242.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 967.1/967.1 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.8/105.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.9 MB/s eta 0:00:00


In [7]:
import hcipy
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

# 1. Setup Fixed Grids (Keep these small for speed during interaction)
pupil_grid = hcipy.make_pupil_grid(256, 1)
focal_grid = hcipy.make_focal_grid(q=4, num_airy=16)
prop = hcipy.FraunhoferPropagator(pupil_grid, focal_grid)

def update_psf(obscuration, num_spiders, spider_width, wavelength):
    # Create the aperture function
    ap_func = hcipy.make_obstructed_circular_aperture(1, obscuration, num_spiders, spider_width)

    # Evaluate and create Wavefront
    pupil_field = hcipy.evaluate_supersampled(ap_func, pupil_grid, 2)
    wf = hcipy.Wavefront(pupil_field, wavelength)

    # Propagate to Focal Plane
    focal_field = prop.forward(wf)
    psf = focal_field.intensity / focal_field.intensity.max()

    # Plotting
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    # Plot Pupil
    hcipy.imshow_field(pupil_field, ax=ax[0], cmap='gray')
    ax[0].set_title("Telescope Pupil")

    # Plot PSF (Log scale to see diffraction spikes better)
    hcipy.imshow_field(np.log10(psf + 1e-6), ax=ax[1], vmin=-5, vmax=0, cmap='inferno')
    ax[1].set_title(f"PSF (Log Scale) at $\lambda$={wavelength}")

    plt.show()

# 2. Create the Interactive Sliders
interact(update_psf,
         obscuration=FloatSlider(value=0.2, min=0, max=1, step=0.05),
         num_spiders=IntSlider(value=4, min=0, max=8, step=1),
         spider_width=FloatSlider(value=0.02, min=0, max=0.1, step=0.005),
         wavelength=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1))

<>:32: SyntaxWarning: invalid escape sequence '\l'
<>:32: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_19915/1813825565.py:32: SyntaxWarning: invalid escape sequence '\l'
  ax[1].set_title(f"PSF (Log Scale) at $\lambda$={wavelength}")


interactive(children=(FloatSlider(value=0.2, description='obscuration', max=1.0, step=0.05), IntSlider(value=4…

<unknown>:43: SyntaxWarning: invalid escape sequence '\l'


<function __main__.update_psf(obscuration, num_spiders, spider_width, wavelength)>